In [6]:
!pip install --upgrade gradio python-dotenv requests -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import gradio as gr
print(gr.__version__)

6.28.0


In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:

import gradio as gr
import requests
import os
from dotenv import load_dotenv

In [11]:
def query_groq(prompt):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}"}
    data = {
        "model": "llama-3.3-70b-versatile",
        "messages": [{"role": "user", "content": prompt}]
    }
    r = requests.post(url, headers=headers, json=data)
    return r.json()["choices"][0]["message"]["content"]

In [12]:
def query_hf(prompt):
    url = "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.3"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}
    r = requests.post(url, headers=headers, json={"inputs": prompt})
    return r.json()[0]["generated_text"]

In [13]:
def explain_code(code, model_choice):
    prompt = f"Explain this code step by step:\n\n{code}"
    try:
        if model_choice == "Groq (Llama 3)":
            return query_groq(prompt)
        else:
            return query_hf(prompt)
    except Exception as e:
        return f"Error: {e}"

In [14]:
def chat_fn(message, history, model_choice):
    reply = explain_code(message, model_choice)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply})
    return "", history

In [16]:
with gr.Blocks() as demo:
    gr.Markdown("## 🧑‍💻 Code Explainer Chatbot")
    model_choice = gr.Radio(["Groq (Llama 3)", "HuggingFace (Mistral)"], value="Groq (Llama 3)", label="Model")
    chatbot = gr.Chatbot()   # <-- key change
    msg = gr.Textbox(placeholder="Paste your code here...")
    clear = gr.Button("Clear")

    msg.submit(chat_fn, [msg, chatbot, model_choice], [msg, chatbot])
    clear.click(lambda: None, None, chatbot)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
